# Naive Early vs. vem Late (Cell + Nucleus)

Same shared embedding as `01_umap_by_condition_over_time.ipynb`, but
instead of comparing conditions frame by frame, pools cells across a
WHOLE range of frames per condition into one single comparison: naive's
early frames vs. vem's late frames (`NAIVE_FRAME_RANGE`/`VEM_FRAME_RANGE`
below - change these to compare different windows). One plot per
embedding method, not paged by frame, since both groups already span
several frames each.

Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs, no manual switch needed.

Requires `main.py` to have been run first.


In [ ]:
import os
import sys

import numpy as np

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/01_naive_vs_vem/notebooks/), so the analysis dir - and config.py -
# is one level up, and the repo root - where shared/ lives - is two more
# levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset, export_plot_data
from shared.density_plots import export_kde


## Plot: embedding with per-window kernel density, naive early vs. vem late

For each embedding method, one PNG and one SVG (in its own `svgs/`
subfolder), written to `output/plots/05_naive_early_vs_vem_late/`.


In [ ]:
# the two frame windows being compared - inclusive on both ends. Change
# these to compare different windows; everything else below adapts
# automatically (group names, legend, file names all derive from these).
NAIVE_FRAME_RANGE = (1, 5)
VEM_FRAME_RANGE   = (10, 13)

NAIVE_GROUP = f'naive (frames {NAIVE_FRAME_RANGE[0]}-{NAIVE_FRAME_RANGE[1]})'
VEM_GROUP   = f'vem (frames {VEM_FRAME_RANGE[0]}-{VEM_FRAME_RANGE[1]})'
GROUP_COLORS = {
    NAIVE_GROUP: 'tab:blue',
    VEM_GROUP:   'tab:red',
}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '05_naive_early_vs_vem_late')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# KDE contour levels (level p encloses the densest (1-p) fraction of a
# group's mass - higher is tighter/closer to the peak) and bandwidth
# multiplier (above 1 smooths over sampling/distance-tie noise)
KDE_LEVELS     = (0.7, 0.8, 0.9)
KDE_BW_ADJUST  = 1

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def label_group(df):
    '''
    Restrict to naive cells within NAIVE_FRAME_RANGE and vem cells
    within VEM_FRAME_RANGE, and label each with a 'group' column -
    every other cell (naive outside its window, vem outside its own,
    or any other condition) is dropped.

    Inputs:
        df: must have: condition, frame
    Outputs:
        df: subset with a 'group' column added (NAIVE_GROUP or VEM_GROUP)
    '''
    conditions = [
        (df['condition'] == config.NAIVE_CONDITION) & df['frame'].between(*NAIVE_FRAME_RANGE),
        (df['condition'] == config.VEM_CONDITION) & df['frame'].between(*VEM_FRAME_RANGE),
    ]
    choices = [NAIVE_GROUP, VEM_GROUP]
    df = df.copy()
    df['group'] = np.select(conditions, choices, default=None)
    return df[df['group'].notna()].reset_index(drop=True)


def process_modality(modality):
    '''
    For each embedding method, load this modality's embedding and
    metadata, restrict to the two frame windows, and export a single
    PDF + SVG comparing them.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        group_dfs: {embedding_method: the two-window subset for this
                    modality and embedding method} (in-memory only)
    '''
    group_dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        group_df = label_group(df)
        print(f"[{modality}/{embedding_method}] {len(group_df)}/{len(df)} cells are in one of the two windows")
        for group in GROUP_COLORS:
            n = (group_df['group'] == group).sum()
            print(f"    {group}: {n} cells")

        export_kde(group_df, 'group', GROUP_COLORS, NOTEBOOK_PLOTS_DIR,
                   f'{modality}_{embedding_method}', embedding_method.upper(),
                   title=f'{MODALITY_NAMES[modality]} morphology: naive early vs. vem late',
                   levels=KDE_LEVELS, bw_adjust=KDE_BW_ADJUST)
        group_dfs[embedding_method] = group_df

    return group_dfs


## Run: cell morphology

In [ ]:
cell_group_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_group_dfs = process_modality('nuc')


## Export plot data

Combines every (modality, embedding method) DataFrame above into one CSV
(`plot_data.csv`, alongside the PNGs/SVGs above) - the same dim_0/dim_1
coordinates and metadata the KDE plots above use, so someone without
this repo can recreate or extend these plots from the raw data alone.

In [ ]:
plot_data = {('cell', m): df for m, df in cell_group_dfs.items()}
plot_data.update({('nuc', m): df for m, df in nuc_group_dfs.items()})
export_plot_data(plot_data, os.path.join(NOTEBOOK_PLOTS_DIR, 'plot_data.csv'))
